# Phase 4 - Notebook 02: Pixel-aligned Gaussian Representation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/02_pixel_aligned_gaussians.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the difference between free-form and pixel-aligned Gaussians
2. Implement depth-to-3D back-projection (the core of pixel alignment)
3. Create pixel-aligned Gaussians from predicted depth and features
4. Merge Gaussians from multiple views into a single scene
5. Understand connections to surfel mapping in SLAM

**Estimated Time**: 60 minutes

**Prerequisites**: Phase 1 (Gaussian model), Notebook 01 (Cost Volume)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyBboxPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

## 1. Free-form vs Pixel-aligned Gaussians

### Original 3DGS (Phase 1): Free-form

```
SfM Point Cloud → Random init → Optimize → Clone/Split/Prune
  - Positions: freely move via gradient descent
  - Density: adaptive (split large, clone small, prune transparent)
  - Count: changes during training (starts ~100K, grows to ~1M+)
```

### Feed-forward (Phase 4): Pixel-aligned

```
Image Pixels → Predict Depth → Back-project to 3D → One Gaussian per pixel
  - Positions: determined by pixel coordinate + predicted depth
  - Density: fixed (H × W Gaussians per input view)
  - Count: constant (e.g., 256×256 = 65,536 per view)
```

In [ ]:
# Side-by-side comparison visualization

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
np.random.seed(42)

# Left: Free-form Gaussians (Phase 1)
ax = axes[0]
ax.set_title('Original 3DGS: Free-form Gaussians', fontsize=13, fontweight='bold')

n = 120
# Clustered random positions simulating scene structure
centers = [(-2, -1), (1, 2), (0, -2), (2, 0)]
for cx, cy in centers:
    nn_pts = n // len(centers)
    px = np.random.randn(nn_pts) * 0.8 + cx
    py = np.random.randn(nn_pts) * 0.8 + cy
    for x, y in zip(px, py):
        sx = np.abs(np.random.randn()) * 0.3 + 0.1
        sy = np.abs(np.random.randn()) * 0.3 + 0.1
        angle = np.random.rand() * 360
        e = Ellipse((x, y), sx, sy, angle=angle, alpha=0.25,
                   facecolor=plt.cm.Set2(np.random.rand()), edgecolor='gray', lw=0.3)
        ax.add_patch(e)

ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.text(0, -3.7, 'Irregular positions, varying density, N changes during training',
        ha='center', fontsize=9, style='italic')
ax.grid(True, alpha=0.15)

# Right: Pixel-aligned Gaussians (Phase 4)
ax = axes[1]
ax.set_title('Feed-forward: Pixel-aligned Gaussians', fontsize=13, fontweight='bold')

grid_n = 16
for i in range(grid_n):
    for j in range(grid_n):
        x = (i - grid_n/2) * 0.45 + np.random.randn() * 0.02
        y = (j - grid_n/2) * 0.45 + np.random.randn() * 0.02
        r = np.sqrt(x**2 + y**2)
        # Scale increases with distance (simulating depth-dependent size)
        s = 0.2 + r * 0.02
        color = plt.cm.viridis(np.clip(1 - r/5, 0.1, 0.9))
        e = Ellipse((x, y), s, s*0.85, angle=np.random.rand()*30,
                   alpha=0.3, facecolor=color, edgecolor='gray', lw=0.2)
        ax.add_patch(e)

ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.text(0, -3.7, 'Regular grid, fixed density, N = H x W per view',
        ha='center', fontsize=9, style='italic')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

## 2. Back-projection: From Pixels to 3D

### 2.1 The Core Operation

Given a pixel $(u, v)$ and its predicted depth $d$, the 3D point is:

$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = d \cdot K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}$$

Expanded:
$$X = \frac{(u - c_x) \cdot d}{f_x}, \quad Y = \frac{(v - c_y) \cdot d}{f_y}, \quad Z = d$$

In [ ]:
def unproject_depth_educational(depth_map, fx, fy, cx, cy):
    """
    Back-project depth map to 3D points.
    Step-by-step educational version.
    """
    H, W = depth_map.shape
    
    # Step 1: Create pixel coordinate grid
    u = torch.arange(W, dtype=torch.float32)
    v = torch.arange(H, dtype=torch.float32)
    vv, uu = torch.meshgrid(v, u, indexing='ij')  # [H, W]
    
    # Step 2: Compute 3D coordinates
    X = (uu - cx) * depth_map / fx
    Y = (vv - cy) * depth_map / fy
    Z = depth_map
    
    # Step 3: Stack into [H, W, 3]
    points_3d = torch.stack([X, Y, Z], dim=-1)
    
    return points_3d


# Create a synthetic depth map
H, W = 32, 32
fx, fy = 30.0, 30.0
cx, cy = W / 2.0, H / 2.0

# Depth with a smooth surface (simulating a wall with bump)
u_grid = torch.linspace(-1, 1, W)
v_grid = torch.linspace(-1, 1, H)
VV, UU = torch.meshgrid(v_grid, u_grid, indexing='ij')
depth_map = 5.0 + 1.0 * torch.sin(UU * np.pi) * torch.cos(VV * np.pi)

# Back-project
points_3d = unproject_depth_educational(depth_map, fx, fy, cx, cy)

print(f"Depth map shape: {depth_map.shape}")
print(f"3D points shape: {points_3d.shape}")
print(f"Total points: {H * W} (one per pixel)")
print(f"X range: [{points_3d[:,:,0].min():.2f}, {points_3d[:,:,0].max():.2f}]")
print(f"Y range: [{points_3d[:,:,1].min():.2f}, {points_3d[:,:,1].max():.2f}]")
print(f"Z range: [{points_3d[:,:,2].min():.2f}, {points_3d[:,:,2].max():.2f}]")

In [ ]:
# Visualize: depth map → 3D point cloud

fig = plt.figure(figsize=(18, 5))

# Depth map
ax1 = fig.add_subplot(131)
im = ax1.imshow(depth_map.numpy(), cmap='plasma')
ax1.set_title('Input: Predicted Depth Map', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax1, fraction=0.046, label='Depth')
ax1.set_xlabel('u (pixels)'); ax1.set_ylabel('v (pixels)')

# 3D point cloud - top view (X-Z)
ax2 = fig.add_subplot(132)
X = points_3d[:,:,0].flatten().numpy()
Y = points_3d[:,:,1].flatten().numpy()
Z = points_3d[:,:,2].flatten().numpy()
sc = ax2.scatter(X, Z, c=Z, cmap='plasma', s=5, alpha=0.7)
ax2.set_xlabel('X'); ax2.set_ylabel('Z (depth)')
ax2.set_title('3D Points (Top View)', fontsize=12, fontweight='bold')
plt.colorbar(sc, ax=ax2, fraction=0.046, label='Depth')
ax2.set_aspect('equal')

# 3D point cloud - front view (X-Y)
ax3 = fig.add_subplot(133)
sc = ax3.scatter(X, -Y, c=Z, cmap='plasma', s=5, alpha=0.7)
ax3.set_xlabel('X'); ax3.set_ylabel('-Y (up)')
ax3.set_title('3D Points (Front View)', fontsize=12, fontweight='bold')
plt.colorbar(sc, ax=ax3, fraction=0.046, label='Depth')
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()
print("Each pixel becomes one 3D point → one Gaussian center.")

## 3. Building Pixel-aligned Gaussians

### 3.1 From 3D Points to Full Gaussians

Each pixel produces a complete Gaussian with:

| Parameter | Source | Activation |
|-----------|--------|------------|
| Position (μ) | Back-projection of (u,v,d) | None |
| Scale (s) | Network prediction | exp (positive) |
| Rotation (q) | Network prediction | normalize (unit quaternion) |
| Opacity (α) | Network prediction | sigmoid ([0,1]) |
| Color (c) | Image pixel RGB | None |

In [ ]:
def create_pixel_aligned_gaussians(depth_map, image_rgb, K,
                                    predicted_scales, predicted_rotations,
                                    predicted_opacities):
    """
    Create pixel-aligned Gaussians from predicted parameters.
    
    Args:
        depth_map: [H, W] predicted depth
        image_rgb: [3, H, W] input image colors
        K: [3, 3] camera intrinsics
        predicted_scales: [3, H, W] predicted log-scales
        predicted_rotations: [4, H, W] predicted quaternions
        predicted_opacities: [1, H, W] predicted logit-opacities
    
    Returns:
        dict of Gaussian parameters
    """
    H, W = depth_map.shape
    N = H * W
    
    # 1. Back-project to 3D positions
    K_inv = torch.inverse(K)
    u = torch.arange(W, dtype=torch.float32)
    v = torch.arange(H, dtype=torch.float32)
    vv, uu = torch.meshgrid(v, u, indexing='ij')
    ones = torch.ones_like(uu)
    pixels = torch.stack([uu, vv, ones], dim=0).reshape(3, -1)  # [3, N]
    
    rays = K_inv @ pixels  # [3, N]
    depth_flat = depth_map.reshape(1, -1)  # [1, N]
    positions = (rays * depth_flat).T  # [N, 3]
    
    # 2. Process scales (apply exp for positive values)
    scales = torch.exp(predicted_scales).permute(1, 2, 0).reshape(N, 3)  # [N, 3]
    
    # 3. Process rotations (normalize to unit quaternion)
    rotations = predicted_rotations.permute(1, 2, 0).reshape(N, 4)
    rotations = F.normalize(rotations, p=2, dim=-1)  # [N, 4]
    
    # 4. Process opacities (sigmoid for [0, 1])
    opacities = torch.sigmoid(predicted_opacities).permute(1, 2, 0).reshape(N, 1)
    
    # 5. Colors from image
    colors = image_rgb.permute(1, 2, 0).reshape(N, 3)  # [N, 3]
    
    return {
        'positions': positions,   # [N, 3]
        'scales': scales,         # [N, 3]
        'rotations': rotations,   # [N, 4]
        'opacities': opacities,   # [N, 1]
        'colors': colors,         # [N, 3]
    }


# Create synthetic predictions
torch.manual_seed(42)
H, W = 16, 16
K = torch.tensor([[30, 0, W/2], [0, 30, H/2], [0, 0, 1]], dtype=torch.float32)

# Simulated network outputs
depth = 4.0 + torch.sin(torch.linspace(-1, 1, W)).unsqueeze(0).expand(H, -1)
image = torch.rand(3, H, W)  # Random image
pred_scales = torch.randn(3, H, W) * 0.5 - 2.0  # Log-scales (small)
pred_rots = torch.randn(4, H, W)  # Raw quaternions
pred_opacities = torch.randn(1, H, W)  # Raw logit-opacities

gaussians = create_pixel_aligned_gaussians(
    depth, image, K, pred_scales, pred_rots, pred_opacities
)

print("Pixel-aligned Gaussians created:")
for key, val in gaussians.items():
    print(f"  {key:12s}: shape={list(val.shape)}, "
          f"range=[{val.min():.3f}, {val.max():.3f}]")
print(f"\nTotal Gaussians: {gaussians['positions'].shape[0]} (= {H} x {W})")

## 4. Multi-View Gaussian Merging

Feed-forward methods process **each input view independently**, producing $H \times W$ Gaussians per view. The final scene is the **union of all views' Gaussians**:

```
View 1: H×W Gaussians ─┐
View 2: H×W Gaussians ──┤── Concatenate ──► Total: N_views × H × W Gaussians
View 3: H×W Gaussians ──┘                        ──► Render from any viewpoint
```

In [ ]:
# Demonstrate multi-view merging

from src.feedforward.pixel_aligned import PixelAlignedGaussians, unproject_depth_to_3d

B, H, W = 1, 8, 8
K_batch = K.unsqueeze(0)  # [1, 3, 3]

views = []
poses = [
    torch.eye(4).unsqueeze(0),  # View 1: identity
    None,                       # View 2: same frame (simplified)
]

for i in range(2):
    depth_i = (3.0 + i * 0.5 + torch.rand(1, 1, H, W) * 0.5)
    features_i = {
        'scales': torch.exp(torch.randn(1, 3, H, W) * 0.3 - 2.0),
        'rotations': F.normalize(torch.randn(1, 4, H, W), p=2, dim=1),
        'opacities': torch.sigmoid(torch.randn(1, 1, H, W)),
    }
    colors_i = torch.rand(1, 3, H, W)
    
    pag = PixelAlignedGaussians.from_depth_and_features(
        depth_i, features_i, K_batch, pose=poses[i], image_colors=colors_i
    )
    views.append(pag)
    print(f"View {i+1}: {pag}")

# Merge
merged = views[0].merge(views[1])
print(f"\nMerged: {merged}")
print(f"Total Gaussians: {views[0].num_gaussians} + {views[1].num_gaussians} = {merged.num_gaussians}")

In [ ]:
# Visualize merged point cloud from two views

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (pag, title, color) in enumerate([
    (views[0], 'View 1 Gaussians', 'blue'),
    (views[1], 'View 2 Gaussians', 'red'),
    (merged, 'Merged Gaussians', None)
]):
    ax = axes[idx]
    pos = pag.positions[0].detach().numpy()  # [N, 3]
    
    if color:
        ax.scatter(pos[:, 0], pos[:, 2], c=color, s=20, alpha=0.6)
    else:
        n1 = views[0].num_gaussians
        ax.scatter(pos[:n1, 0], pos[:n1, 2], c='blue', s=15, alpha=0.5, label='View 1')
        ax.scatter(pos[n1:, 0], pos[n1:, 2], c='red', s=15, alpha=0.5, label='View 2')
        ax.legend()
    
    ax.set_title(f'{title} (N={pag.num_gaussians})', fontsize=11, fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Z')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Multi-view merging: simply concatenate Gaussians from all views.")
print("The renderer handles overlapping Gaussians via alpha blending (Phase 1).")

## 5. Connection to Surfel Mapping (SLAM)

Pixel-aligned Gaussians are closely related to **surfel mapping** in SLAM:

| Property | Surfel (SLAM) | Pixel-aligned Gaussian |
|----------|--------------|------------------------|
| Geometry | Oriented disk | 3D ellipsoid |
| Center | Back-project depth | Back-project predicted depth |
| Orientation | Surface normal | Rotation quaternion |
| Size | Pixel footprint | Network-predicted scale |
| Color | RGB from image | RGB from image |
| Confidence | Observation count | Predicted opacity |
| Update | Incremental fusion | Single-shot prediction |

The key difference: surfels are **fused over time** (SLAM), while pixel-aligned Gaussians are **predicted in one shot** (feed-forward).

In [ ]:
# Summary

summary = """
╔═══════════════════════════════════════════════════════════════════════╗
║         Notebook 02 Summary: Pixel-aligned Gaussians                 ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FREE-FORM vs PIXEL-ALIGNED                                       ║
║     - Free-form: random init, optimized, adaptive density            ║
║     - Pixel-aligned: structured grid, predicted, fixed count         ║
║                                                                      ║
║  2. BACK-PROJECTION                                                  ║
║     - P_3d = depth * K^{-1} * [u, v, 1]^T                          ║
║     - Each pixel → one 3D Gaussian center                           ║
║                                                                      ║
║  3. GAUSSIAN PARAMETERS                                              ║
║     - Position: from back-projection                                ║
║     - Scale: exp(network output) → positive                         ║
║     - Rotation: normalize(network output) → unit quaternion         ║
║     - Opacity: sigmoid(network output) → [0, 1]                    ║
║     - Color: from input image directly                              ║
║                                                                      ║
║  4. MULTI-VIEW MERGING                                               ║
║     - Process each view independently                               ║
║     - Concatenate Gaussians from all views                          ║
║     - Total: N_views × H × W Gaussians                             ║
║                                                                      ║
║  5. SLAM CONNECTION                                                  ║
║     - Similar to surfel mapping (depth → disk/ellipsoid)            ║
║     - But single-shot prediction vs incremental fusion              ║
║                                                                      ║
╚═══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

## What's Next?

**[03_mvsplat_architecture.ipynb](./03_mvsplat_architecture.ipynb)** - Complete MVSplat architecture deep dive

---

## References

1. MVSplat: https://arxiv.org/abs/2403.14627
2. pixelSplat: https://arxiv.org/abs/2312.12337
3. ElasticFusion (surfel mapping): https://arxiv.org/abs/1502.01762